In [ ]:
from datasets import load_dataset

ds = load_dataset("Nan-Do/code-search-net-python")

README.md:   0%|          | 0.00/2.65k [00:00<?, ?B/s]

data/train-00000-of-00004-ee77a7de79eb2a(…): reconstructing file:   0%|          |  0.00B /  155MB            

data/train-00000-of-00004-ee77a7de79eb2a(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-648b3bede2edf6(…): reconstructing file:   0%|          |  0.00B /  139MB            

data/train-00001-of-00004-648b3bede2edf6(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-1dfd72b171e6b2(…): reconstructing file:   0%|          |  0.00B /  153MB            

data/train-00002-of-00004-1dfd72b171e6b2(…): downloading bytes:           |  0.00B            

data/train-00003-of-00004-184ab6d0e3c690(…): reconstructing file:   0%|          |  0.00B /  151MB            

data/train-00003-of-00004-184ab6d0e3c690(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/455243 [00:00<?, ? examples/s]

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url', 'partition', 'summary'],
        num_rows: 455243
    })
})

In [ ]:
ds['train']['original_string'][0]

'def train(train_dir, model_save_path=None, n_neighbors=None, knn_algo=\'ball_tree\', verbose=False):\n    """\n    Trains a k-nearest neighbors classifier for face recognition.\n\n    :param train_dir: directory that contains a sub-directory for each known person, with its name.\n\n     (View in source code to see train_dir example tree structure)\n\n     Structure:\n        <train_dir>/\n        ├── <person1>/\n        │   ├── <somename1>.jpeg\n        │   ├── <somename2>.jpeg\n        │   ├── ...\n        ├── <person2>/\n        │   ├── <somename1>.jpeg\n        │   └── <somename2>.jpeg\n        └── ...\n\n    :param model_save_path: (optional) path to save model on disk\n    :param n_neighbors: (optional) number of neighbors to weigh in classification. Chosen automatically if not specified\n    :param knn_algo: (optional) underlying data structure to support knn.default is ball_tree\n    :param verbose: verbosity of training\n    :return: returns knn classifier that was trained on 

In [ ]:
print(repr(ds['train']["code"][0]))

'def train(train_dir, model_save_path=None, n_neighbors=None, knn_algo=\'ball_tree\', verbose=False):\n    """\n    Trains a k-nearest neighbors classifier for face recognition.\n\n    :param train_dir: directory that contains a sub-directory for each known person, with its name.\n\n     (View in source code to see train_dir example tree structure)\n\n     Structure:\n        <train_dir>/\n        ├── <person1>/\n        │   ├── <somename1>.jpeg\n        │   ├── <somename2>.jpeg\n        │   ├── ...\n        ├── <person2>/\n        │   ├── <somename1>.jpeg\n        │   └── <somename2>.jpeg\n        └── ...\n\n    :param model_save_path: (optional) path to save model on disk\n    :param n_neighbors: (optional) number of neighbors to weigh in classification. Chosen automatically if not specified\n    :param knn_algo: (optional) underlying data structure to support knn.default is ball_tree\n    :param verbose: verbosity of training\n    :return: returns knn classifier that was trained on 

In [ ]:
from collections import Counter
print(Counter(ds['train']['partition']))

Counter({'train': 410175, 'valid': 22977, 'test': 22091})


In [ ]:
import io, tokenize

def tokenize_code(source):
    tokens = []
    prev_toktype = tokenize.INDENT
    try:
        for tok in tokenize.generate_tokens(io.StringIO(source).readline):
            tok_type, tok_str = tok.type, tok.string

            if tok_type in (tokenize.COMMENT, tokenize.ENDMARKER):
                continue

            # docstrings
            if tok_type == tokenize.STRING and prev_toktype in (
                tokenize.INDENT, tokenize.NEWLINE, tokenize.DEDENT
            ):
                prev_toktype = tok_type
                continue

            if tok_type == tokenize.INDENT:
                tokens.append("<INDENT>")
            elif tok_type == tokenize.DEDENT:
                tokens.append("<DEDENT>")
            elif tok_type in (tokenize.NEWLINE, tokenize.NL):
                # collapse blank lines / removed comments / removed docstrings
                if tokens and tokens[-1] not in ("<NEWLINE>", "<INDENT>", "<DEDENT>"):
                    tokens.append("<NEWLINE>")
            else:
                tokens.append(tok_str)
            prev_toktype = tok_type
        return tokens
    except (IndentationError, tokenize.TokenError, TabError):
        return None

In [ ]:
records = []
for code, part in zip(ds['train']['code'], ds['train']['partition']):
    toks = tokenize_code(code)
    if toks:
        records.append((toks, part))

train_data = [t for t, p in records if p == 'train']
valid_data = [t for t, p in records if p == 'valid']
test_data  = [t for t, p in records if p == 'test']
print(len(train_data), len(valid_data), len(test_data))

410014 22970 22088


building vocabulary with train data

In [ ]:
vocab={}
from collections import Counter
vocab['<PAD>']=0
vocab['<UNK>']=1
vocab['<BOS>']=2
vocab['<EOS>']=3

counter=Counter()
for token_list in train_data:
  counter.update(token for token in token_list)

for token,freq in counter.most_common(50000):
  if freq>=2:
    vocab[token]=len(vocab)

In [ ]:
import pickle

In [ ]:
with open('vocab.pkl','wb') as f:
  pickle.dump(vocab,f)

encode tokens

In [ ]:
def encode(token_list):
  return [vocab[token] if token in vocab else vocab['<UNK>'] for token in token_list]

In [ ]:
max_func_len=1024

In [ ]:
encoded_train = [encode(token) for token in train_data if len(token)<=max_func_len]
encoded_test = [encode(token) for token in test_data if len(token)<=max_func_len]
encoded_valid = [encode(token) for token in valid_data if len(token)<=max_func_len]

In [ ]:
print(len(encoded_train))
print(len(encoded_test))
print(len(encoded_valid))

407515
21986
22792


In [ ]:
lengths = [len(row) for row in encoded_train]

print("Average:", sum(lengths) / len(lengths))
print("Max:", max(lengths))
print("Min:", min(lengths))

Average: 129.5526569574126
Max: 1024
Min: 23


In [ ]:
import pickle
with open('encoded_train.pkl','wb') as f:
  pickle.dump(encoded_train,f)

with open('encoded_test.pkl','wb') as f:
  pickle.dump(encoded_test,f)

with open('encoded_valid.pkl','wb') as f:
  pickle.dump(encoded_valid,f)